In [ ]:
"""
Lightweight test harness for the pooled rideshare simulation.

How to use
---------
1. Make sure the classes SimulationLogger, Stop, Request, Route, Driver, Simulation
   from your code are available in this module's global scope.

   - Easiest: paste this whole block *below* your existing class definitions
     in the same file.

   - OR: put this in a separate file and add at the top:
       from your_module_name import Simulation, Request, Route, Driver

2. In a Python shell or in your main, call:

       run_all_tests()

   or call individual tests like:

       test_single_request_completes()
"""

import os
import csv
import tempfile
import contextlib
from datetime import datetime, timedelta

import networkx as nx
import osmnx as ox


# ==========================================================
# Utility: temporary working directory
# ==========================================================

@contextlib.contextmanager
def temporary_cwd():
    """
    Context manager that creates a temporary directory and
    changes the current working directory into it for the
    duration of the context.
    """
    old_cwd = os.getcwd()
    with tempfile.TemporaryDirectory() as tmp:
        os.chdir(tmp)
        try:
            yield tmp
        finally:
            os.chdir(old_cwd)


# ==========================================================
# Utility: toy graph + ToySimulation subclass
# ==========================================================

def make_toy_graph():
    """
    Create a tiny toy road network graph with lat/lon so we don't
    download Manhattan for tests.

    3 nodes laid roughly along a line:
        N1 (40.75, -73.99)
        N2 (40.76, -73.98)
        N3 (40.77, -73.97)
    Fully connected with simple 'length' attributes.
    """
    G = nx.MultiDiGraph()
    G.graph["crs"] = "EPSG:4326"  # lat/lon CRS

    nodes = {
        1: {"y": 40.75, "x": -73.99},
        2: {"y": 40.76, "x": -73.98},
        3: {"y": 40.77, "x": -73.97},
    }

    for nid, attrs in nodes.items():
        G.add_node(nid, **attrs)

    # Connect nodes both directions with simple lengths
    edges = [
        (1, 2, 1000.0),
        (2, 3, 1000.0),
        (1, 3, 2000.0),
    ]
    for u, v, length in edges:
        G.add_edge(u, v, length=length)
        G.add_edge(v, u, length=length)

    return G


class ToySimulation(Simulation):
    """
    A Simulation subclass that uses the toy graph above instead of
    loading/downloading the Manhattan graph. This keeps tests fast
    and deterministic.
    """

    def initialize(self):
        # Initialize logger (writes into current working dir)
        self.logger.initialize()

        # Use toy graph instead of Manhattan
        self.G_latlon = make_toy_graph()
        self.G_proj = ox.project_graph(self.G_latlon)

        # Same initial driver logic as original Simulation
        current_hour = self.start_time.hour
        target_drivers = self.driver_schedule.get(current_hour, 5)
        self.add_drivers(target_drivers)


# ==========================================================
# Helpers to check route correctness
# ==========================================================

def assert_pickup_before_dropoff(route: Route):
    """
    Ensure that, in a given Route, each request's pickup occurs
    before its dropoff in the optimal sequence.
    Raises AssertionError if violated.
    """
    df = route.get_optimal_time()
    if df.empty:
        return
    for req_id in df["Request ID"].unique():
        sub = df[df["Request ID"] == req_id]
        if len(sub) < 2:
            # Possibly only one of the stops is visited in this fragment.
            continue
        pickup_row = sub[sub["Type"] == "PICKUP"].iloc[0]
        dropoff_row = sub[sub["Type"] == "DROPOFF"].iloc[0]
        if not (pickup_row["Step"] < dropoff_row["Step"]):
            raise AssertionError(
                f"Request {req_id} has dropoff before pickup in route!"
            )


# ==========================================================
# Test 1: Single driver, single request (basic sanity)
# ==========================================================

def test_single_request_completes():
    """
    One driver, one request: request should be completed, with
    actual pickup and dropoff times logged.
    """
    print("Running test_single_request_completes...")
    with temporary_cwd():
        base_time = datetime(2025, 1, 1, 8, 0, 0)

        req_data = [
            {
                "id": "R1",
                "time": base_time + timedelta(minutes=1),
                "count": 1,
                "p_lat": 40.75,
                "p_lon": -73.99,
                "d_lat": 40.77,
                "d_lon": -73.97,
            }
        ]

        schedule_data = {8: 1}  # one driver

        sim = ToySimulation(
            start=base_time,
            end=base_time + timedelta(minutes=40),
            request_data=req_data,
            driver_schedule=schedule_data,
            shift_hours=2,
            start_locations=[(40.75, -73.99)],
            vehicle_capacity_types=[4],
        )

        sim.initialize()
        sim.run_simulation()

        assert os.path.exists("request_logs.csv")
        assert os.path.exists("driver_logs.csv")

        with open("request_logs.csv") as f:
            reader = csv.DictReader(f)
            rows = list(reader)

        if not (len(rows) == 1):
            raise AssertionError("Expected exactly one request in logs.")
        r = rows[0]
        if r["Request_ID"] != "R1":
            raise AssertionError("Unexpected Request_ID in logs.")
        if r["Status"] != "COMPLETED":
            raise AssertionError("Request R1 did not complete.")
        if not r["Act_Pickup_Time"] or not r["Act_Dropoff_Time"]:
            raise AssertionError("Actual pickup/dropoff times missing.")

    print("  ✅ test_single_request_completes passed.")


# ==========================================================
# Test 2: Capacity pre-check (no overload)
# ==========================================================

def test_capacity_respected_over_time():
    """
    R1 has 3 passengers, R2 has 2, capacity is 4.
    The system *may* accept both requests, but it must ensure that
    at no time are more than 4 passengers onboard simultaneously.
    We check that using the actual pickup/dropoff times in the log.
    """
    print("Running test_capacity_respected_over_time...")
    with temporary_cwd():
        base_time = datetime(2025, 1, 1, 8, 0, 0)

        # R1: 3 passengers, R2: 2 passengers, capacity=4
        req_data = [
            {
                "id": "R1",
                "time": base_time + timedelta(minutes=1),
                "count": 3,
                "p_lat": 40.75,
                "p_lon": -73.99,
                "d_lat": 40.77,
                "d_lon": -73.97,
            },
            {
                "id": "R2",
                "time": base_time + timedelta(minutes=2),
                "count": 2,
                "p_lat": 40.75,
                "p_lon": -73.99,
                "d_lat": 40.76,
                "d_lon": -73.98,
            },
        ]

        schedule_data = {8: 1}

        sim = ToySimulation(
            start=base_time,
            end=base_time + timedelta(minutes=40),
            request_data=req_data,
            driver_schedule=schedule_data,
            shift_hours=2,
            start_locations=[(40.75, -73.99)],
            vehicle_capacity_types=[4],  # capacity = 4
        )

        sim.initialize()
        sim.run_simulation()

        # Read requests and reconstruct intervals
        with open("request_logs.csv") as f:
            reader = csv.DictReader(f)
            rows = list(reader)

        completed_ids = {row["Request_ID"] for row in rows}
        # We *allow* both to be completed
        if "R1" not in completed_ids:
            raise AssertionError("R1 should be completed.")
        if "R2" not in completed_ids:
            raise AssertionError("R2 should be completed.")

        # Build pickup/dropoff events
        events = []  # list of (time, delta_passengers)
        capacity = 4

        for row in rows:
            count = int(row["Passengers"])
            if row["Act_Pickup_Time"] and row["Act_Dropoff_Time"]:
                t_pick = datetime.strptime(row["Act_Pickup_Time"], "%Y-%m-%d %H:%M:%S")
                t_drop = datetime.strptime(row["Act_Dropoff_Time"], "%Y-%m-%d %H:%M:%S")
                events.append((t_pick, +count))
                events.append((t_drop, -count))

        # Sort events by time; in case of ties, process dropoffs before pickups
        events.sort(key=lambda e: (e[0], 1 if e[1] > 0 else 0))

        load = 0
        max_load = 0
        for t, delta in events:
            load += delta
            max_load = max(max_load, load)

        if max_load > capacity:
            raise AssertionError(
                f"Capacity violated: max concurrent passengers {max_load} > capacity {capacity}"
            )

    print("  ✅ test_capacity_respected_over_time passed.")


# ==========================================================
# Test 3: Pooling (UPDATE_ROUTE and both IDs in sequence)
# ==========================================================

def test_pooling_creates_update_route():
    """
    Two small requests should be pooled onto the same driver.
    Check that UPDATE_ROUTE appears and includes both request IDs.
    """
    print("Running test_pooling_creates_update_route...")
    with temporary_cwd():
        base_time = datetime(2025, 1, 1, 8, 0, 0)

        req_data = [
            {
                "id": "R1",
                "time": base_time + timedelta(minutes=1),
                "count": 1,
                "p_lat": 40.75,
                "p_lon": -73.99,
                "d_lat": 40.77,
                "d_lon": -73.97,
            },
            {
                "id": "R2",
                "time": base_time + timedelta(minutes=2),
                "count": 1,
                "p_lat": 40.76,
                "p_lon": -73.98,
                "d_lat": 40.77,
                "d_lon": -73.97,
            },
        ]

        schedule_data = {8: 1}

        sim = ToySimulation(
            start=base_time,
            end=base_time + timedelta(minutes=40),
            request_data=req_data,
            driver_schedule=schedule_data,
            shift_hours=2,
            start_locations=[(40.75, -73.99)],
            vehicle_capacity_types=[4],
        )

        sim.initialize()
        sim.run_simulation()

        # Both should be completed
        with open("request_logs.csv") as f:
            r_reader = csv.DictReader(f)
            r_rows = list(r_reader)

        completed_ids = {r["Request_ID"] for r in r_rows}
        if not {"R1", "R2"}.issubset(completed_ids):
            raise AssertionError("Expected both R1 and R2 to complete.")

        # Look for UPDATE_ROUTE containing both IDs
        with open("driver_logs.csv") as f:
            d_reader = csv.DictReader(f)
            d_rows = list(d_reader)

        update_rows = [row for row in d_rows if row["Category"] == "UPDATE_ROUTE"]
        if not update_rows:
            raise AssertionError("No UPDATE_ROUTE events found.")

        pooled_found = False
        for row in update_rows:
            seq = row["Route_Sequence"]
            if "R1" in seq and "R2" in seq:
                pooled_found = True
                break

        if not pooled_found:
            raise AssertionError(
                "Expected an UPDATE_ROUTE route sequence containing both R1 and R2."
            )

    print("  ✅ test_pooling_creates_update_route passed.")


# ==========================================================
# Test 4: Route feasibility (pickup before dropoff)
# ==========================================================

def test_pickup_before_dropoff_in_route():
    """
    Build a Route directly and ensure the MILP respects
    pickup-before-dropoff constraints.
    """
    print("Running test_pickup_before_dropoff_in_route...")
    with temporary_cwd():
        G_latlon = make_toy_graph()
        base_time = datetime(2025, 1, 1, 8, 0, 0)

        # Two requests on the toy network
        r1 = Request(
            "R1",
            base_time,
            1,
            40.75,
            -73.99,
            40.77,
            -73.97,
        )
        r2 = Request(
            "R2",
            base_time,
            1,
            40.76,
            -73.98,
            40.77,
            -73.97,
        )

        stops = [r1.pickup_loc, r1.dropoff_loc, r2.pickup_loc, r2.dropoff_loc]

        route = Route(
            G_latlon,
            stops,
            vehicle_start_coords=(40.75, -73.99),
            start_time=base_time,
            speed_mph=20,
            vehicle_capacity=4,
            initial_load=0,
        )

        if not route.generate_route():
            raise AssertionError("Route MILP failed to generate a feasible route.")

        # Raises if constraint violated
        assert_pickup_before_dropoff(route)

    print("  ✅ test_pickup_before_dropoff_in_route passed.")


# ==========================================================
# Test 5: Shift end & OFF_DUTY logs
# ==========================================================

def test_shift_end_and_off_duty_logged():
    """
    With shift_hours=1 and simulation over 2+ hours, driver should:
      - Have SHIFT_START at beginning,
      - SHIFT_END_PENDING when shift is over,
      - OFF_DUTY once idle and removed.
    """
    print("Running test_shift_end_and_off_duty_logged...")
    with temporary_cwd():
        base_time = datetime(2025, 1, 1, 8, 0, 0)

        req_data = [
            {
                "id": "R1",
                "time": base_time + timedelta(minutes=1),
                "count": 1,
                "p_lat": 40.75,
                "p_lon": -73.99,
                "d_lat": 40.76,
                "d_lon": -73.98,
            }
        ]

        schedule_data = {8: 1, 9: 0, 10: 0}

        sim = ToySimulation(
            start=base_time,
            end=base_time + timedelta(hours=2, minutes=10),
            request_data=req_data,
            driver_schedule=schedule_data,
            shift_hours=1,
            start_locations=[(40.75, -73.99)],
            vehicle_capacity_types=[4],
        )

        sim.initialize()
        sim.run_simulation()

        with open("driver_logs.csv") as f:
            d_reader = csv.DictReader(f)
            d_rows = list(d_reader)

        categories = [row["Category"] for row in d_rows]
        if "SHIFT_START" not in categories:
            raise AssertionError("SHIFT_START not logged.")
        if "SHIFT_END_PENDING" not in categories:
            raise AssertionError("SHIFT_END_PENDING not logged.")
        if "OFF_DUTY" not in categories:
            raise AssertionError("OFF_DUTY not logged.")

    print("  ✅ test_shift_end_and_off_duty_logged passed.")


# ==========================================================
# Test 6: No requests (request log has only header)
# ==========================================================

def test_no_requests_results_in_empty_request_log():
    """
    If there are no requests at all, request_logs.csv should exist and
    only contain the header row.
    """
    print("Running test_no_requests_results_in_empty_request_log...")
    with temporary_cwd():
        base_time = datetime(2025, 1, 1, 8, 0, 0)

        req_data = []  # no requests
        schedule_data = {8: 1}

        sim = ToySimulation(
            start=base_time,
            end=base_time + timedelta(minutes=20),
            request_data=req_data,
            driver_schedule=schedule_data,
            shift_hours=1,
            start_locations=[(40.75, -73.99)],
            vehicle_capacity_types=[4],
        )

        sim.initialize()
        sim.run_simulation()

        if not os.path.exists("request_logs.csv"):
            raise AssertionError("request_logs.csv not found.")

        with open("request_logs.csv") as f:
            reader = csv.reader(f)
            rows = list(reader)

        if len(rows) != 1:
            raise AssertionError(
                f"Expected 1 row (header) in request_logs.csv, got {len(rows)} rows."
            )

    print("  ✅ test_no_requests_results_in_empty_request_log passed.")


# ==========================================================
# Test 7: Logging format sanity
# ==========================================================

def test_logging_format():
    """
    Check that request_logs.csv has the expected columns and that
    date fields are parseable when present.
    """
    print("Running test_logging_format...")
    with temporary_cwd():
        base_time = datetime(2025, 1, 1, 8, 0, 0)

        req_data = [
            {
                "id": "R1",
                "time": base_time + timedelta(minutes=1),
                "count": 1,
                "p_lat": 40.75,
                "p_lon": -73.99,
                "d_lat": 40.76,
                "d_lon": -73.98,
            }
        ]

        schedule_data = {8: 1}

        sim = ToySimulation(
            start=base_time,
            end=base_time + timedelta(minutes=30),
            request_data=req_data,
            driver_schedule=schedule_data,
            shift_hours=1,
            start_locations=[(40.75, -73.99)],
            vehicle_capacity_types=[4],
        )

        sim.initialize()
        sim.run_simulation()

        with open("request_logs.csv") as f:
            reader = csv.DictReader(f)
            rows = list(reader)
            fieldnames = set(reader.fieldnames or [])

        expected_cols = {
            "Request_ID",
            "Request_Time",
            "Status",
            "Passengers",
            "Pickup_Lat",
            "Pickup_Lon",
            "Dropoff_Lat",
            "Dropoff_Lon",
            "Exp_Pickup_Time",
            "Exp_Dropoff_Time",
            "Act_Pickup_Time",
            "Act_Dropoff_Time",
        }
        if fieldnames != expected_cols:
            raise AssertionError(
                f"Unexpected request_logs columns: {fieldnames} "
                f"(expected {expected_cols})"
            )

        # Check time fields parse when non-empty
        for row in rows:
            for col in [
                "Request_Time",
                "Exp_Pickup_Time",
                "Exp_Dropoff_Time",
                "Act_Pickup_Time",
                "Act_Dropoff_Time",
            ]:
                if row[col]:
                    datetime.strptime(row[col], "%Y-%m-%d %H:%M:%S")

    print("  ✅ test_logging_format passed.")


# ==========================================================
# Convenience: run all tests in one call
# ==========================================================

def run_all_tests():
    """
    Run all test functions defined in this file.
    This is NOT called automatically; you call it yourself.

    Example:
        from your_module import run_all_tests
        run_all_tests()
    """
    print("=== Running all rideshare simulation tests ===")
    tests = [
        test_single_request_completes,
        test_capacity_respected_over_time,    # <-- updated here
        test_pooling_creates_update_route,
        test_pickup_before_dropoff_in_route,
        test_shift_end_and_off_duty_logged,
        test_no_requests_results_in_empty_request_log,
        test_logging_format,
    ]

    passed = 0
    failed = 0

    for t in tests:
        try:
            t()
            passed += 1
        except Exception as e:
            failed += 1
            print(f"  ❌ {t.__name__} FAILED: {e}")

    print("=== Test summary ===")
    print(f"  Passed: {passed}")
    print(f"  Failed: {failed}")
    print("====================")


In [ ]:
run_all_tests()

---

In [ ]:
import unittest
import osmnx as ox
import networkx as nx
from datetime import datetime, timedelta
import warnings
import math

# Suppress warnings for clean output
warnings.filterwarnings("ignore")

class TestComprehensiveRideshare(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        """
        Load map ONCE for all tests.
        Radius 1500m ensures we have enough space for "Detour" tests.
        """
        print("\n--- Initializing Map & Environment ---")
        # 1.5km radius around Penn Station
        cls.G_latlon = ox.graph_from_point((40.7505, -73.9934), dist=1500, network_type='drive')
        cls.G_proj = ox.project_graph(cls.G_latlon)
        cls.start_time = datetime(2025, 1, 1, 8, 0, 0)
        
        # --- Defined Locations ---
        # 1. Central Hub (Penn Station)
        cls.loc_penn = (40.7505, -73.9934)
        # 2. Nearby (MSG - 200m away)
        cls.loc_msg = (40.7503, -73.9910)
        # 3. Far Away (Hudson Yards/Javits - 1km+ away)
        cls.loc_far = (40.7580, -74.0000)

    def setUp(self):
        """Reset Driver and Logger before EVERY test"""
        self.logger = SimulationLogger()
        self.logger.d_writer = unittest.mock.Mock()
        self.logger.r_writer = unittest.mock.Mock()
        
        # Standard Driver: Capacity 4, Starts at Penn Station
        self.driver = Driver("D1", self.loc_penn, self.start_time, self.G_latlon, self.G_proj, capacity=4)

    # ==================================================================
    # GROUP 1: The "Happy Paths" (Sanity Checks)
    # ==================================================================
    
    def test_01_basic_trip_lifecycle(self):
        """Verify a passenger goes from PENDING -> ON_BOARD -> COMPLETED."""
        print("\nTest 01: Basic Lifecycle...")
        req = Request("R1", self.start_time, 1, self.loc_penn[0], self.loc_penn[1], self.loc_msg[0], self.loc_msg[1])
        
        # Force assignment
        stops = [req.pickup_loc, req.dropoff_loc]
        route = Route(self.G_latlon, stops, self.loc_penn, self.start_time, 20, 4)
        route.generate_route()
        self.driver.accept_request(req, route, self.logger)
        
        # 1. Check Pickup (Immediate)
        self.driver.status_update(self.start_time, self.logger)
        self.assertEqual(req.status, 'ON_BOARD', "Failed to pickup passenger at start location.")
        
        # 2. Check Dropoff (Future)
        arrival = req.expected_dropoff_time + timedelta(minutes=1)
        self.driver.status_update(arrival, self.logger)
        self.assertEqual(req.status, 'COMPLETED', "Failed to complete trip.")

    def test_02_shuttle_logic(self):
        """Verify we can shuttle groups (2+2 in a 3-seater)."""
        print("Test 02: Shuttle Logic (Dynamic Capacity)...")
        # Use a small car (Capacity 3)
        self.driver.capacity = 3
        
        # Two groups of 2 (Total 4 people)
        rA = Request("GrpA", self.start_time, 2, self.loc_penn[0], self.loc_penn[1], self.loc_msg[0], self.loc_msg[1])
        rB = Request("GrpB", self.start_time, 2, self.loc_penn[0], self.loc_penn[1], self.loc_msg[0], self.loc_msg[1])
        
        # Should succeed by dropping A before picking B
        route = self.driver.request_validity_check(rA, self.start_time) # Accept A
        self.driver.accept_request(rA, route, self.logger)
        
        route_combined = self.driver.request_validity_check(rB, self.start_time) # Try adding B
        
        self.assertIsNotNone(route_combined, "Driver rejected valid shuttle opportunity.")
        # Verify load never exceeds 3 inside the optimization result
        for stop in route_combined.optimal_sequence:
            # (Approximate check logic handled by Gurobi, simply getting a route proves it found a valid path)
            pass

    # ==================================================================
    # GROUP 2: The "Guards" (Constraint Verification)
    # ==================================================================

    def test_03_strict_capacity_overflow(self):
        """Verify 5 people are rejected from a 4-seater."""
        print("Test 03: Clown Car Rejection...")
        req = Request("R_Huge", self.start_time, 5, self.loc_penn[0], self.loc_penn[1], self.loc_msg[0], self.loc_msg[1])
        
        # Should return None
        route = self.driver.request_validity_check(req, self.start_time)
        self.assertIsNone(route, "Driver accepted a group larger than vehicle capacity!")

    def test_04_initial_load_rejection(self):
        """Verify we respect passengers already in the car."""
        print("Test 04: Initial Load Rejection...")
        # Simulating: Driver has 3 people in car already
        self.driver.passengers_on_board = 3
        
        # Try to pick up 2 more (Total 5 > 4)
        req = Request("R_New", self.start_time, 2, self.loc_penn[0], self.loc_penn[1], self.loc_msg[0], self.loc_msg[1])
        
        route = self.driver.request_validity_check(req, self.start_time)
        self.assertIsNone(route, "Driver ignored initial load and overfilled car.")

    # ==================================================================
    # GROUP 3: Business Logic (Quality of Service)
    # ==================================================================

    def test_05_detour_policy_check(self):
        """
        CRITICAL: Verify the 1.2x detour limit.
        We must ensure the original trip is > 60 seconds for the check to trigger.
        """
        print("Test 05: Detour/Delay Policy Guard...")
        
        # 1. Assign Passenger A (Medium Trip: Penn -> Far)
        # Dist: ~1km. Speed: 20mph. Time: ~2-3 mins.
        # This ensures old_duration > 60s, enabling the protection logic.
        reqA = Request("R_Medium", self.start_time, 1, self.loc_penn[0], self.loc_penn[1], self.loc_far[0], self.loc_far[1])
        
        routeA = Route(self.G_latlon, [reqA.pickup_loc, reqA.dropoff_loc], self.loc_penn, self.start_time, 20, 4)
        routeA.generate_route()
        self.driver.accept_request(reqA, routeA, self.logger)
        
        # 2. New Request B (Massive Detour)
        # Request B requires driving to a completely different location BEFORE dropping A.
        # We create a request that is very far away.
        # Coordinates for "Battery Park" (approx 5km south)
        loc_battery = (40.7033, -74.0170)
        
        reqB = Request("R_Disruptive", self.start_time, 1, loc_battery[0], loc_battery[1], self.loc_penn[0], self.loc_penn[1])
        
        # 3. Check Validity
        # Original Trip: ~3 mins
        # New Trip (Penn -> Battery -> Penn -> Far): ~30 mins
        # 30 mins > (3 mins * 1.2). Rejection expected.
        result_route = self.driver.request_validity_check(reqB, self.start_time)
        
        self.assertIsNone(result_route, "Driver accepted a detour that violated the 1.2x delay policy.")
        print("   -> Driver correctly rejected the massive detour.")

    # ==================================================================
    # GROUP 4: Edge Case Stability
    # ==================================================================

    def test_06_zero_distance_request(self):
        """Verify the system doesn't crash if Pickup == Dropoff."""
        print("Test 06: Zero-Distance Stability...")
        req = Request("R_Zero", self.start_time, 1, self.loc_penn[0], self.loc_penn[1], self.loc_penn[0], self.loc_penn[1])
        
        # Should probably be valid (instant dropoff) or rejected, BUT NOT CRASH.
        try:
            route = self.driver.request_validity_check(req, self.start_time)
            # We don't strictly care if it returns None or Route, just no Exception.
            print("   -> Handled zero-distance gracefully.")
        except Exception as e:
            self.fail(f"System crashed on zero-distance request: {e}")

if __name__ == '__main__':
    import unittest.mock
    # Run logic for Jupyter
    unittest.main(argv=['first-arg-is-ignored'], exit=False, verbosity=1)